# QLoRA Fine-tuning: Qwen2.5-7B for Vietnamese Invoice Extraction

Fine-tune Qwen2.5-7B-Instruct with QLoRA (4-bit) on Google Colab T4 GPU.

**Goal**: Improve structured JSON extraction from Vietnamese OCR invoice text.

**Hardware**: Google Colab T4 (16GB VRAM) — fits with 4-bit quantization

In [ ]:
# Cell 1: Install required packages
!pip install -q transformers peft bitsandbytes datasets accelerate trl torch
!pip install -q sentencepiece protobuf
print('Setup complete')

In [ ]:
# Cell 2: Model and training configuration
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
BITS = 4  # QLoRA 4-bit
MAX_SEQ_LEN = 2048
EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
OUTPUT_DIR = "/content/qwen2.5-invoice-lora"

print(f"Model: {MODEL_NAME}")
print(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Quantization: {BITS}-bit QLoRA")
print(f"Training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}")

In [ ]:
# Cell 3: Training dataset — Vietnamese invoice instruction-following pairs
import json

train_data = [
    {
        "instruction": "Trích xuất thông tin từ hóa đơn sau dạng JSON:",
        "input": "HÓA ĐƠN\nNgày: 15/01/2024\nCông ty ABC\nMã số thuế: 0312345678\nDịch vụ tư vấn\nThành tiền: 5.000.000đ\nThuế VAT 10%: 500.000đ\nTổng: 5.500.000đ",
        "output": json.dumps({"vendor": "Công ty ABC", "tax_id": "0312345678", "date": "2024-01-15", "subtotal": 5000000, "vat": 500000, "total": 5500000, "line_items": [{"name": "Dịch vụ tư vấn", "amount": 5000000}]}, ensure_ascii=False)
    },
    {
        "instruction": "Trích xuất thông tin từ hóa đơn sau dạng JSON:",
        "input": "HÓA ĐƠN GIÁ TRỊ GIA TĂNG\nSố: 0000456\nNgày: 20/03/2024\nCông ty Công nghệ XYZ\nMST: 0109876543\nPhần mềm kế toán: 8.500.000đ\nCài đặt: 1.500.000đ\nCộng: 10.000.000đ\nVAT 10%: 1.000.000đ\nTổng thanh toán: 11.000.000đ",
        "output": json.dumps({"vendor": "Công ty Công nghệ XYZ", "tax_id": "0109876543", "date": "2024-03-20", "invoice_number": "0000456", "subtotal": 10000000, "vat": 1000000, "total": 11000000, "line_items": [{"name": "Phần mềm kế toán", "amount": 8500000}, {"name": "Cài đặt", "amount": 1500000}]}, ensure_ascii=False)
    },
    {
        "instruction": "Trích xuất thông tin từ hóa đơn sau dạng JSON:",
        "input": "HÓA ĐƠN\nNgày 05/02/2024\nNguyễn Văn A\nDịch vụ thiết kế website\nGiá: 15.000.000 VNĐ\nVAT: 1.500.000 VNĐ\nTổng: 16.500.000 VNĐ",
        "output": json.dumps({"vendor": "Nguyễn Văn A", "tax_id": null, "date": "2024-02-05", "subtotal": 15000000, "vat": 1500000, "total": 16500000, "line_items": [{"name": "Dịch vụ thiết kế website", "amount": 15000000}]}, ensure_ascii=False)
    },
    {
        "instruction": "Trích xuất thông tin từ hóa đơn sau dạng JSON:",
        "input": "INVOICE\nDate: 10/04/2024\nSupplier: Công ty TNHH Thương mại Hà Nội\nTax code: 0101234567\nOffice supplies: 2.300.000\nPrinting services: 700.000\nTotal before tax: 3.000.000\nVAT 10%: 300.000\nTotal: 3.300.000",
        "output": json.dumps({"vendor": "Công ty TNHH Thương mại Hà Nội", "tax_id": "0101234567", "date": "2024-04-10", "subtotal": 3000000, "vat": 300000, "total": 3300000, "line_items": [{"name": "Office supplies", "amount": 2300000}, {"name": "Printing services", "amount": 700000}]}, ensure_ascii=False)
    },
    {
        "instruction": "Trích xuất thông tin từ hóa đơn sau dạng JSON:",
        "input": "HÓA ĐƠN ĐIỆN TỬ\nKý hiệu: BB/24P\nSố: 0001234\nNgày lập: 28/06/2024\nNgười bán: Công ty Cổ phần Giải pháp Số\nMST: 0312987654\nDịch vụ Cloud hosting 6 tháng: 24.000.000\nThuế GTGT (10%): 2.400.000\nTổng tiền thanh toán: 26.400.000",
        "output": json.dumps({"vendor": "Công ty Cổ phần Giải pháp Số", "tax_id": "0312987654", "date": "2024-06-28", "invoice_number": "0001234", "serial": "BB/24P", "subtotal": 24000000, "vat": 2400000, "total": 26400000, "line_items": [{"name": "Dịch vụ Cloud hosting 6 tháng", "amount": 24000000}]}, ensure_ascii=False)
    },
    {
        "instruction": "Trích xuất thông tin từ hóa đơn sau dạng JSON:",
        "input": "HÓA ĐƠN\n12/07/2024\nCty TNHH Vận tải Miền Nam\nMST: 0314567890\nVận chuyển hàng hóa HCM-HN: 4.500.000\nPhụ phí: 200.000\nCộng: 4.700.000\nVAT: 470.000\nTổng: 5.170.000",
        "output": json.dumps({"vendor": "Công ty TNHH Vận tải Miền Nam", "tax_id": "0314567890", "date": "2024-07-12", "subtotal": 4700000, "vat": 470000, "total": 5170000, "line_items": [{"name": "Vận chuyển hàng hóa HCM-HN", "amount": 4500000}, {"name": "Phụ phí", "amount": 200000}]}, ensure_ascii=False)
    },
    {
        "instruction": "Trích xuất thông tin từ hóa đơn sau dạng JSON:",
        "input": "HÓA ĐƠN GTGT\nSố: 2024/08/0089\nNgày: 18/08/2024\nBên bán: Trung tâm Đào tạo Tin học ABC\nMã số thuế: 0204567890\nKhóa học Python cơ bản: 3.600.000\nTài liệu học: 400.000\nTổng trước thuế: 4.000.000\nThuế VAT 10%: 400.000\nThành tiền: 4.400.000",
        "output": json.dumps({"vendor": "Trung tâm Đào tạo Tin học ABC", "tax_id": "0204567890", "date": "2024-08-18", "invoice_number": "2024/08/0089", "subtotal": 4000000, "vat": 400000, "total": 4400000, "line_items": [{"name": "Khóa học Python cơ bản", "amount": 3600000}, {"name": "Tài liệu học", "amount": 400000}]}, ensure_ascii=False)
    },
    {
        "instruction": "Trích xuất thông tin từ hóa đơn sau dạng JSON:",
        "input": "PHIẾU THU\nNgày: 30/09/2024\nCông ty Quảng cáo Sáng Tạo\nMST: 0315678901\nDịch vụ quảng cáo Facebook 3 tháng: 18.000.000\nThiết kế banner: 2.000.000\nCộng tiền hàng: 20.000.000\nThuế 10%: 2.000.000\nTổng cộng: 22.000.000",
        "output": json.dumps({"vendor": "Công ty Quảng cáo Sáng Tạo", "tax_id": "0315678901", "date": "2024-09-30", "subtotal": 20000000, "vat": 2000000, "total": 22000000, "line_items": [{"name": "Dịch vụ quảng cáo Facebook 3 tháng", "amount": 18000000}, {"name": "Thiết kế banner", "amount": 2000000}]}, ensure_ascii=False)
    },
    {
        "instruction": "Trích xuất thông tin từ hóa đơn sau dạng JSON:",
        "input": "HÓA ĐƠN\nNgày 02/11/2024\nNhà hàng Phở Hà Nội 1946\nMST: 0108765432\nPhở bò đặc biệt x5: 350.000\nNước ngọt x5: 75.000\nTổng: 425.000\nKhông VAT (nhà hàng nhỏ)",
        "output": json.dumps({"vendor": "Nhà hàng Phở Hà Nội 1946", "tax_id": "0108765432", "date": "2024-11-02", "subtotal": 425000, "vat": 0, "total": 425000, "line_items": [{"name": "Phở bò đặc biệt x5", "amount": 350000}, {"name": "Nước ngọt x5", "amount": 75000}]}, ensure_ascii=False)
    },
    {
        "instruction": "Trích xuất thông tin từ hóa đơn sau dạng JSON:",
        "input": "HÓA ĐƠN ĐIỆN TỬ\nMã: INV-2024-1205-001\nNgày: 05/12/2024\nCông ty Phần mềm Việt\nMST: 0316789012\nGói Enterprise ERP 1 năm: 120.000.000\nTriển khai và đào tạo: 30.000.000\nHỗ trợ kỹ thuật 1 năm: 12.000.000\nTổng dịch vụ: 162.000.000\nVAT 10%: 16.200.000\nTổng thanh toán: 178.200.000",
        "output": json.dumps({"vendor": "Công ty Phần mềm Việt", "tax_id": "0316789012", "date": "2024-12-05", "invoice_number": "INV-2024-1205-001", "subtotal": 162000000, "vat": 16200000, "total": 178200000, "line_items": [{"name": "Gói Enterprise ERP 1 năm", "amount": 120000000}, {"name": "Triển khai và đào tạo", "amount": 30000000}, {"name": "Hỗ trợ kỹ thuật 1 năm", "amount": 12000000}]}, ensure_ascii=False)
    }
]

print(f"Training examples loaded: {len(train_data)}")
print("Sample input:", train_data[0]["input"][:100])

In [ ]:
# Cell 4: Load Qwen2.5-7B with 4-bit quantization
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit quantization config for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading tokenizer from {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loading model with {BITS}-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"Model loaded. Parameters: {model.num_parameters()/1e9:.1f}B")
print(f"Memory footprint: {model.get_memory_footprint()/1e9:.2f} GB")

In [ ]:
# Cell 5: Configure LoRA adapter with PEFT
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for k-bit training (enables gradient checkpointing)
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 6: SFT training with TRL SFTTrainer
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# Format data as chat template
def format_chat(example):
    """Format training examples as Qwen chat format."""
    messages = [
        {"role": "system", "content": "Bạn là trợ lý AI chuyên trích xuất thông tin từ hóa đơn tiếng Việt. Hãy trả về JSON hợp lệ."},
        {"role": "user", "content": f"{example['instruction']}\n\n{example['input']}"},
        {"role": "assistant", "content": example["output"]}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

# Build dataset (in production, use 200+ examples)
dataset = Dataset.from_list(train_data)
dataset = dataset.map(format_chat)
print(f"Dataset prepared: {len(dataset)} examples")
print("Sample formatted text (first 200 chars):")
print(dataset[0]["text"][:200])

# Training arguments
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=5,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="none",
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
)

# Initialize SFT trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 7: Save LoRA adapter
import os

adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"LoRA adapter saved to: {adapter_path}")
print("Files saved:")
for f in os.listdir(adapter_path):
    size = os.path.getsize(os.path.join(adapter_path, f)) / 1e6
    print(f"  {f}: {size:.1f} MB")

In [ ]:
# Cell 8: Test inference with fine-tuned model
from peft import PeftModel

def extract_invoice(ocr_text, model, tokenizer, max_new_tokens=512):
    """Run invoice extraction inference."""
    messages = [
        {"role": "system", "content": "Bạn là trợ lý AI chuyên trích xuất thông tin từ hóa đơn tiếng Việt. Hãy trả về JSON hợp lệ."},
        {"role": "user", "content": f"Trích xuất thông tin từ hóa đơn sau dạng JSON:\n\n{ocr_text}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the generated tokens
    generated = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

# Test on a new invoice not in training data
test_invoice = """HÓA ĐƠN
Ngày: 15/01/2025
Công ty Dệt May Tiến Thành
MST: 0400123456
Vải cotton 50m: 7.500.000đ
Chỉ may 10 cuộn: 200.000đ
Tổng: 7.700.000đ
VAT 10%: 770.000đ
Thanh toán: 8.470.000đ"""

result = extract_invoice(test_invoice, model, tokenizer)
print("=== Fine-tuned Model Output ===")
print(result)

In [ ]:
# Cell 9: Before/after comparison — base model vs fine-tuned
from transformers import pipeline
import json

test_cases = [
    {
        "name": "Invoice with missing tax_id",
        "text": "HÓA ĐƠN\n02/02/2025\nCửa hàng Điện tử Minh Châu\nTivi Samsung 55'': 12.000.000\nVAT 10%: 1.200.000\nTổng: 13.200.000",
        "expected_keys": ["vendor", "date", "total", "vat"]
    },
    {
        "name": "Multi-line items invoice",
        "text": "HÓA ĐƠN GTGT\n20/02/2025\nCty TNHH Văn phòng phẩm Sao Mai\nMST: 0312000111\nGiấy A4: 500.000\nMực in: 800.000\nBút bi (hộp): 300.000\nTổng: 1.600.000\nVAT: 160.000\nThành tiền: 1.760.000",
        "expected_keys": ["vendor", "tax_id", "date", "total", "line_items"]
    }
]

print("=" * 60)
print("BEFORE vs AFTER COMPARISON")
print("=" * 60)

for tc in test_cases:
    print(f"\nTest: {tc['name']}")
    print(f"Input: {tc['text'][:80]}...")
    
    # Fine-tuned model output
    ft_output = extract_invoice(tc["text"], model, tokenizer)
    print(f"Fine-tuned output: {ft_output}")
    
    # Check JSON validity and key coverage
    try:
        parsed = json.loads(ft_output)
        found_keys = [k for k in tc["expected_keys"] if k in parsed]
        accuracy = len(found_keys) / len(tc["expected_keys"])
        print(f"JSON valid: YES | Key coverage: {accuracy*100:.0f}% ({found_keys})")
    except json.JSONDecodeError:
        print("JSON valid: NO — output is not valid JSON")

print("\n=== Comparison complete ===")